### Word2Vec Implementation from Scratch

# Introduction
 Word2Vec is a popular technique for word embeddings, which captures the meaning of words by placing them in a continuous vector space.
 In this exercise, you will implement Word2Vec using NumPy and complete the missing parts of the code.
We will represent each word as a one-hot vector, meaning each word in the vocabulary is mapped to a unique binary vector with only one active (1) position.

import necessary libraries

In [1]:
import numpy as np
from collections import defaultdict

c:\Users\niush\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\niush\anaconda3\lib\site-packages\numpy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
c:\Users\niush\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
settings = {
	'window_size': 7,
	'n': 10,
	'epochs': 50,
	'learning_rate': 0.01,
	'embed_size': 50
}

In [3]:
class word2vec:
    def __init__(self, settings):
        """
        Initialize the Word2Vec model with given hyperparameters.
        """

        self.batch_size = settings['n']
        self.lr = settings['learning_rate']
        self.epochs = settings['epochs']
        self.window_size = settings['window_size']
        self.embed_size = settings['embed_size']
        self.words_list = None
        self.word_index = None
        self.index_word = None
        self.v_count = None
        self.w1 = None
        self.w2 = None

    def get_vocab(self, corpus):
        
        word_counts = defaultdict(int)

        for word in corpus:
            word_counts[word] += 1

        vocab_size = len(word_counts)
        words_list = list(word_counts.keys())
        
        word2idx = {word: i for i, word in enumerate(words_list)}
        idx2word = {i: word for i, word in enumerate(words_list)}
        
        return words_list, vocab_size, word2idx, idx2word
    
    # Function to get previous words in the window
    def prev_words(self, i, doc):
        out = []
        window = self.window_size//2
        for index in range(i - window, i):
            if index >= 0:
                out.append(self.word_index[doc[index]])
            else:
                out.append(0)  # 0 = <PAD>
        return out

    # Function to get next words in the window
    def next_words(self, i, doc):
        out = []
        window = self.window_size//2
        for index in range(i + 1, i + window + 1, 1):
            if index < len(doc):
                out.append(self.word_index[doc[index]])
            else:
                out.append(0)  # 0 = <PAD>
        return out
    
    def generate_training_data(self, corpus):
        
        # part 1
        self.words_list, self.v_count, self.word_index, self.index_word = self.get_vocab(corpus)
        
        # part 2
        X = []
        y = []
        for index, word in enumerate(corpus):
            prev = self.prev_words(index, corpus)
            next = self.next_words(index, corpus)
            assert len(prev) == len(next)
            X.append(prev + next)
            y.append([self.word_index[word]])
        
        X = np.array(X)
        y = np.array(y).flatten()

        training_data = list(zip(X, y))
        
        return training_data

    def word2onehot(self, word):
        
        word_vec = np.zeros(self.v_count)
        word_vec[self.word_index[word]] = 1
        return word_vec
    
    def train(self, training_data):

        # Initialize weights randomly
        self.w1 = np.random.rand(self.v_count, self.embed_size)
        self.w2 = np.random.rand(self.embed_size, self.v_count)

        for epoch in range(self.epochs):
            total_loss = 0

            # Initialize gradients and batch counter
            batch_grad_w1 = np.zeros_like(self.w1)
            batch_grad_w2 = np.zeros_like(self.w2)
            batch_count = 0

            for i, (x, target) in enumerate(training_data):

                # Generate one-hot encoded vectors for context words (x)
                context_vectors = [self.word2onehot(self.index_word[word_index]) for word_index in x]

                # Compute the average of context vectors to form the input vector (CBOW style)
                input_vector = np.mean(context_vectors, axis=0)

                # Generate one-hot encoded vector for the target word
                y_true = self.word2onehot(self.index_word[target])

                # Forward pass
                y_predicted, hidden_layer_output, _ = self.forward_pass(input_vector)

                # Call the backpropagation function for training
                loss, batch_grad_w1, batch_grad_w2, batch_count = self.backpropagation(
                    input_vector, y_true, hidden_layer_output, y_predicted,
                    batch_grad_w1, batch_grad_w2, batch_count,
                    training_complete=(i + 1 == len(training_data))
                )
                
                # Accumulate total loss
                total_loss += loss

            # Average loss over the entire training dataset
            total_loss /= len(training_data)
            print(f'Epoch {epoch+1}/{self.epochs} - Loss: {total_loss}')

    def softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)

    def forward_pass(self, input_vector):
        hidden_layer_output = np.dot(self.w1.T, input_vector)
        output_scores = np.dot(self.w2.T, hidden_layer_output)
        predicted_probabilities = self.softmax(output_scores)
        return predicted_probabilities, hidden_layer_output, output_scores
    
    def backpropagation(self, input_vector, y_true, hidden_layer_output, y_predicted, 
                         batch_grad_w1, batch_grad_w2, batch_count, training_complete=False):
    
        # Step 1: Calculate the error (difference between predicted and true labels)
        error = y_predicted - y_true
        loss = -np.log(np.dot(y_true, y_predicted) + 1e-9)
        
        # Step 2: Calculate gradients using backpropagation
        gradient_w2 = np.outer(hidden_layer_output, error)
        gradient_w1 = np.outer(input_vector, np.dot(self.w2, error))
        
        # Step 3: Accumulate gradients for Batch Processing
        batch_grad_w1 += gradient_w1
        batch_grad_w2 += gradient_w2
        batch_count += 1
        
        # Step 4: Update weights if batch is completed or training is finished
        if training_complete or (batch_count == self.batch_size):
            # Average the gradients for batch processing
            batch_grad_w1 /= batch_count
            batch_grad_w2 /= batch_count

            # Update weights using Gradient Descent
            self.w1 -= self.lr * batch_grad_w1
            self.w2 -= self.lr * batch_grad_w2

            # Reset gradients and batch count for the next batch
            batch_grad_w1 = np.zeros_like(self.w1)
            batch_grad_w2 = np.zeros_like(self.w2)
            batch_count = 0
        
        return loss, batch_grad_w1, batch_grad_w2, batch_count

    def word_vec(self, word):
        word_index = self.word_index[word]
        word_embedding = self.w1[word_index]
        return word_embedding 

    def vec_sim(self, word, top_n):
        v_w1 = self.word_vec(word)
        similarity = {}

        for other_word in self.words_list:
            if other_word == word:
                continue
            v_w2 = self.word_vec(other_word)
            sim = np.dot(v_w1, v_w2) / (np.linalg.norm(v_w1) * np.linalg.norm(v_w2))
            similarity[other_word] = sim

        words_sorted = sorted(similarity.items(), key=lambda x: x[1], reverse=True)

        for word, sim in words_sorted[:top_n]:
            print(f'{word}: {sim}')


In [4]:
text = "Natural language processing and machine learning open up fascinating possibilities, allowing machines to analyze,\
 understand, and respond to human language in ways that were once thought impossible."

In [5]:
corpus = [word.lower() for word in text.split()]
print(len(corpus))
print(corpus)

27
['natural', 'language', 'processing', 'and', 'machine', 'learning', 'open', 'up', 'fascinating', 'possibilities,', 'allowing', 'machines', 'to', 'analyze,', 'understand,', 'and', 'respond', 'to', 'human', 'language', 'in', 'ways', 'that', 'were', 'once', 'thought', 'impossible.']


In [6]:
w2v = word2vec(settings)

training_data = w2v.generate_training_data(corpus)

In [7]:
w2v.train(training_data)

Epoch 1/50 - Loss: 3.6609346489802665
Epoch 2/50 - Loss: 3.6424875089445297
Epoch 3/50 - Loss: 3.6246698710759335
Epoch 4/50 - Loss: 3.6074454492613452
Epoch 5/50 - Loss: 3.590781137707574
Epoch 6/50 - Loss: 3.5746466676503985
Epoch 7/50 - Loss: 3.559014306016599
Epoch 8/50 - Loss: 3.5438585903321163
Epoch 9/50 - Loss: 3.529156095024977
Epoch 10/50 - Loss: 3.5148852249865397
Epoch 11/50 - Loss: 3.501026032854347
Epoch 12/50 - Loss: 3.4875600569843948
Epoch 13/50 - Loss: 3.4744701775067766
Epoch 14/50 - Loss: 3.4617404882195393
Epoch 15/50 - Loss: 3.449356182382037
Epoch 16/50 - Loss: 3.437303450730255
Epoch 17/50 - Loss: 3.4255693902593896
Epoch 18/50 - Loss: 3.4141419225099185
Epoch 19/50 - Loss: 3.4030097202570024
Epoch 20/50 - Loss: 3.3921621416438827
Epoch 21/50 - Loss: 3.3815891709211106
Epoch 22/50 - Loss: 3.3712813650581306
Epoch 23/50 - Loss: 3.361229805584151
Epoch 24/50 - Loss: 3.351426055093626
Epoch 25/50 - Loss: 3.34186211791958
Epoch 26/50 - Loss: 3.332530404537051
Epoch 

In [8]:
word = "machine"
vec = w2v.word_vec(word)
print(word, vec)

# Find similar words
w2v.vec_sim("machine", 3)

machine [0.47128402 0.09215993 0.3800065  0.88900151 0.0848901  0.88740245
 0.22391808 0.11345262 0.29486463 0.47966952 0.49270187 0.87890195
 0.60756618 0.82269273 0.04269681 0.94088444 0.05298453 0.11138544
 0.52549421 0.74929264 0.77090149 0.83898809 0.13514378 0.13183065
 0.2105941  0.24822484 0.16641846 0.70137556 0.97962198 0.32493313
 0.00601746 0.28811809 0.33008607 0.35916059 0.93759225 0.40887677
 0.6461673  0.36506538 0.6840244  0.35784748 0.50304952 0.81262393
 0.52317893 0.54661989 0.54258299 0.43631928 0.06668501 0.71854717
 0.08537145 0.67142396]
understand,: 0.8233599079446274
thought: 0.7961920358304846
and: 0.7941930588764198
